# ColdSite-DTI — speed test: normal vs mixed precision (Colab, T4)

**The question.** KIBA cells for HyperAttentionDTI and MolTrans are projected at 12–14 h
each in normal (fp32) precision — longer than one Kaggle commit. Mixed precision uses the
T4's tensor cores and might roughly halve that. This measures it instead of guessing.

For each of the four models, on real KIBA training rows, with the grid's batch sizes and
optimisers, it times training steps in three settings — **fp32** (what DAVIS used),
**fp32 + cudnn autotuning**, and **mixed precision + autotuning** — and reports the
speed-up, peak GPU memory, whether mixed precision produced any bad (non-finite) step, how
far it moves the model's outputs, and the projected KIBA hours per cell.

**Nothing is trained for real and no trainer is changed.** About 15 minutes.

## Before you run

**Runtime -> Change runtime type -> T4 GPU**, then run the cells in order. Section 2 asks
you to authorise Drive — that click is yours. The result table is saved to
`MyDrive/coldsite-speed-test/`, where Claude can read it directly.


## 1. Check the GPU

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print('GPU  :', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)


## 2. Mount Drive (where the result is saved)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/coldsite-speed-test'
os.makedirs(OUT, exist_ok=True)
print('results ->', OUT)


## 3. Clone the repo

In [ ]:
REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
SRC = '/content/ColdSite-DTI_New'
if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q subword-nmt
assert os.path.exists('src/model/benchmark_speed.py'), 'This checkout predates the speed test -- re-run this cell.'
!git log --oneline -1


## 4. Data and the KIBA splits

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
!python -m src.data.build_splits 2>&1 | grep -E 'kiba|leakage'

import pandas as pd
got = tuple(len(pd.read_csv(f'data/splits/kiba/random/{p}.csv')) for p in ('train', 'valid', 'test'))
assert got == (82778, 11825, 23651), f'KIBA random split differs from the project: {got}'
print('KIBA random split matches.')


## 5. Run the speed test

Streams one line per model and setting, then the tables. If a model runs out of GPU memory,
that is itself worth knowing — send the error.

In [ ]:
!python -m src.model.benchmark_speed --dataset kiba --split random --steps 50 --warmup 10 --out {OUT}


## 6. The result

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/speed_kiba.md').read()))
